# Initialization

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Checking data integration

In [0]:
TABLE_CONFIG = [
    "bronze.crm_cust_info",
    "bronze.crm_prd_info",
    "bronze.crm_sales_details",
    "bronze.erp_cust_az12",
    "bronze.erp_loc_a101",
    "bronze.erp_px_cat_g1v2"
]

for table in TABLE_CONFIG:
    spark.read.table(table).limit(50).display()



# Read Table

In [0]:
df = spark.read.table("bronze.crm_prd_info")

# Check is there any null and duplicate values on prd_id

In [0]:
window = Window.partitionBy("prd_id")

df.select("*",
          count("prd_id").over(window).alias("number_of_prd_id"))\
          .filter((col("prd_id").isNull()) | (col("number_of_prd_id") != 1)).display()
# We dont have any

# Check is there any null and duplicate values on prd_key

In [0]:
window = Window.partitionBy("prd_key")

df.select("*",
          count("prd_key").over(window).alias("number_of_prd_key"))\
          .filter((col("prd_key").isNull()) | (col("number_of_prd_key") > 1)).display()

there are a lot of historical data, we will keep it since their prd_id is different and to evade historical lost for old sales data

# Trimming all column

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
       df =  df.withColumn(field.name, trim(col(field.name)))
df.display()

# Retrieve the category_id from prd_key

In [0]:
df = (df.withColumn("prd_cat_id", substring(col("prd_key"), 1, 5))
      .withColumn("prd_cat_id", regexp_replace(col("prd_cat_id"), "-", "_"))
      .withColumn("prd_key", substring(col("prd_key"), 7, length(col("prd_key")) - 6))
      )
df.display()

# Handling null values on prd_cost

In [0]:
df = df.fillna(0, subset = ["prd_cost"])

# Standardizing data on prd_line

In [0]:
df.select("prd_line").distinct().display()

In [0]:
df = df.withColumn( "prd_line",
    when(col("prd_line") == "R", "Road")
    .when(col("prd_line") == "M", "Mountain")
    .when(col("prd_line") == "T", "Touring")
    .when(col("prd_line") == "S", "Other Sales")
    .otherwise("N/A")
)

# Check the date data

In [0]:
df.select("*").filter((col("prd_start_dt") > col("prd_end_dt")) | (col("prd_start_dt").isNull()) | (col("prd_end_dt").isNull())).display()

- the product with null prd_end_dt suggesting the product still aired, so we will keep the null in prd_end_dt
- no null data in prd_start_dt
- there are some problem in the historical record, where the prd_start_dt > prd_end_dt, which doesn't make any sense

# Clean the date data

In [0]:
window = Window.partitionBy("prd_key").orderBy("prd_start_dt")

df = df.withColumn("prd_end_dt",lead("prd_start_dt").over(window))

# Sanity Check

In [0]:
df.limit(100).display()